# Notebook 5: a 13-head multitask network, warm-started

The one alternative model with direct testimony behind it. The entry at live rank 48
calls their 17-head multitask fine-tune "the single largest ensemble contributor we
found."

It also resolves a contradiction that has shaped this project. The same entry says
**freeze, don't fine-tune**, and both are true. Fine-tuning a pretrained encoder on this
challenge's four labels alone hurts, because 1,285 to 2,335 labels drag the
representation toward a small sample. Fine-tuning on those four *plus* a dozen auxiliary
targets works, because the auxiliary heads hold the encoder in place. Every failed
fine-tune in our notes, CheMeleon and MoLFormer and ChemBERTa, was the first kind.

**Thirteen heads**, against their seventeen; we are missing only their four ChEMBL ones.

| head group | heads | compounds |
| :--- | ---: | ---: |
| challenge pIC50 | 4 | 4,905 (1,285 to 2,335 labelled each) |
| single-concentration screen | 4 | 4,375 |
| qHTS continuous potency, AID 1851 | 5 | 16,559 |

Union is about 21,000 molecules. Targets are missing at random across them, so the loss
is masked: each molecule contributes only to the heads it actually has.

**Why this one and not a seventh encoder.** Both arms we ship are fixed features handed
to a tabular model. This trains the representation itself, so its errors should differ,
which is the property that pays in a blend. Our own earlier multitask attempt failed at
-7.43%, but that was a cold-start MLP on descriptors with no pretrained encoder and no
auxiliary data.

**Honest risk.** This is the most likely of anything we have built to simply not work.
Watch the control: a warm-started encoder should beat a randomly initialised one from the
first epoch, and if it does not, the checkpoint is not loading into the model properly.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/cyp_results'
except Exception as e:
    print('Drive unavailable:', type(e).__name__)
os.makedirs(OUTDIR, exist_ok=True)
print('writing to', OUTDIR)

In [ ]:
!pip -q install chemprop==2.3.1

In [ ]:
import gc, json, time, urllib.request, numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

CACHE=Path('/content/cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
meta=pd.read_csv(CACHE/'meta_train.csv.gz'); mete=pd.read_csv(CACHE/'meta_test.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy'); G=np.load(CACHE/'scaffold_groups.npy',allow_pickle=True)
AUX=np.load(CACHE/'aux_train.npy')
SM_TR=meta['SMILES'].to_numpy(); SM_TE=mete['SMILES'].to_numpy()

CK='/content/chemprop_medium.pt'
if not Path(CK).exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/jeremycheminf/'
        'openadmet_scripts/main/CYP_Challenge/checkpoints/chemprop_medium.pt', CK)
print('checkpoint', round(Path(CK).stat().st_size/1e6,1), 'MB')

URL=('https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?query=download'
     '&record_type=datatable&actvty=all&response_type=save&aid=1851')
if not Path('/content/aid1851.csv').exists():
    urllib.request.urlretrieve(URL, '/content/aid1851.csv')
raw=pd.read_csv('/content/aid1851.csv', low_memory=False, skiprows=[1])
raw['pac50']=-pd.to_numeric(raw['Fit_LogAC50'], errors='coerce')
PANEL={'p450-cyp1a2':'CYP1A2','p450-cyp2c9':'CYP2C9','p450-cyp2c19':'CYP2C19',
       'p450-cyp2d6':'CYP2D6','p450-cyp3a4':'CYP3A4'}
raw['panel']=raw['Panel Name'].map(PANEL)
raw=raw.dropna(subset=['panel','pac50','PUBCHEM_EXT_DATASOURCE_SMILES'])
print('qHTS measurements', len(raw))

## Build the 13-column target table

Public compounds that are also challenge compounds are dropped, so nothing the blind set
contains leaks in through an auxiliary head.

In [ ]:
def canon(s):
    m=Chem.MolFromSmiles(s) if isinstance(s,str) else None
    return Chem.MolToSmiles(m) if m is not None else None

ours={c for c in map(canon, np.concatenate([SM_TR,SM_TE])) if c}
pub=raw.groupby(['PUBCHEM_EXT_DATASOURCE_SMILES','panel'],as_index=False).pac50.median()
pub['canon']=[canon(s) for s in pub.PUBCHEM_EXT_DATASOURCE_SMILES]
pub=pub.dropna(subset=['canon'])
pub=pub[~pub.canon.isin(ours)]
wide=pub.pivot_table(index='PUBCHEM_EXT_DATASOURCE_SMILES',columns='panel',values='pac50')
PAN=['CYP1A2','CYP2C9','CYP2C19','CYP2D6','CYP3A4']
wide=wide.reindex(columns=PAN)
print('public compounds after dropping overlaps', len(wide))

HEADS=[f'chal_{i}' for i in ISO]+[f'scr_{i}' for i in ISO]+[f'qh_{p}' for p in PAN]
n_ch=len(SM_TR); n_pub=len(wide)
SM_ALL=np.concatenate([SM_TR, wide.index.to_numpy()])
Y=np.full((n_ch+n_pub, len(HEADS)), np.nan, np.float32)
for j,i in enumerate(ISO):
    Y[:n_ch, j]=meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float)
    Y[:n_ch, 4+j]=AUX[:, j]
Y[n_ch:, 8:]=wide.to_numpy(float)
M=np.isfinite(Y)
# Normalisation from development rows only; using all rows would fit a statistic on
# held-out labels. Affine, so it cannot inflate a Spearman, but it should still be clean.
_norm = Y.copy(); _norm[np.flatnonzero(hold)] = np.nan
mu = np.nanmean(_norm, 0); sd = np.nanstd(_norm, 0)
Yz_nan=((Y-mu)/np.where(sd>0,sd,1.0)).astype(np.float32)  # NaN kept as the mask
print('table', Y.shape, '| labels per head:', dict(zip(HEADS, M.sum(0).tolist())))

In [ ]:
from chemprop import data, featurizers
from chemprop.models.model import MPNN

FZ = featurizers.SimpleMoleculeMolGraphFeaturizer()

# Targets live inside the datapoints, so chemprop's own loader batches them with the
# graphs and nothing has to be re-featurised per minibatch. NaN marks "not measured"
# and becomes the loss mask.
PTS_ALL = [data.MoleculeDatapoint.from_smi(s, Yz_nan[i])
           for i, s in enumerate(SM_ALL)]
PTS_TE = [data.MoleculeDatapoint.from_smi(s, np.zeros(len(HEADS), np.float32))
          for s in SM_TE]
DS_TE = data.MoleculeDataset(PTS_TE, FZ)

def loader(idx=None, bs=128, shuffle=False, test=False):
    pts = PTS_TE if test else ([PTS_ALL[i] for i in idx] if idx is not None else PTS_ALL)
    return data.build_dataloader(data.MoleculeDataset(pts, FZ), num_workers=0,
                                 batch_size=bs, shuffle=shuffle, drop_last=False)

def _fp(m, bmg, V_d, X_d):
    # BatchMolGraph.to() moves in place and returns None, unlike a torch tensor, so it
    # must be called as a statement rather than used as an expression.
    if hasattr(bmg, 'to'):
        bmg.to(DEVICE)
    V_d = V_d.to(DEVICE) if torch.is_tensor(V_d) else V_d
    X_d = X_d.to(DEVICE) if torch.is_tensor(X_d) else X_d
    return m.fingerprint(bmg, V_d, X_d)

class Multi(nn.Module):
    def __init__(self, ckpt, n_out, warm=True):
        super().__init__()
        m = MPNN.load_from_file(ckpt)
        if not warm:
            for p in m.parameters():
                nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.zeros_(p)
        self.enc = m
        with torch.no_grad():                      # probe on CPU, before .to(DEVICE)
            bmg, V_d, X_d, *_ = next(iter(loader([0, 1, 2, 3], bs=4)))
            d = m.fingerprint(bmg, V_d, X_d).shape[1]
        self.head = nn.Sequential(nn.Linear(d, 512), nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(512, n_out))

    def forward(self, bmg, V_d, X_d):
        return self.head(_fp(self.enc, bmg, V_d, X_d))

def train(tr_idx, epochs=25, warm=True, lr=3e-4, bs=128, seed=0):
    torch.manual_seed(seed)
    m = Multi(CK, len(HEADS), warm).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-5)
    ld = loader(tr_idx, bs=bs, shuffle=True)
    for ep in range(epochs):
        m.train()
        for batch in ld:
            bmg, V_d, X_d, Yb, *_ = batch
            t = Yb.to(DEVICE).float()
            msk = torch.isfinite(t)
            if msk.sum() == 0:
                continue
            opt.zero_grad()
            p = m(bmg, V_d, X_d)
            loss = (((p - torch.nan_to_num(t)) ** 2) * msk).sum() / msk.sum()
            loss.backward()
            opt.step()
    return m

@torch.no_grad()
def predict(m, idx=None, bs=256, test=False):
    m.eval()
    out = [m(*batch[:3]).cpu().numpy() for batch in loader(idx, bs=bs, test=test)]
    return np.vstack(out) * sd + mu

## Control first: does the warm start actually load

Two short runs on the same fold, one from the checkpoint and one from a reinitialised
encoder. The warm start should be clearly ahead. If it is not, the checkpoint is not
reaching the weights and everything below is meaningless.

In [ ]:
y1=meta['CYP3A4_pIC50_direct_inhibition'].to_numpy(float)
obs=np.isfinite(y1); dev=np.flatnonzero(obs & ~hold); ho=np.flatnonzero(obs & hold)
pub_idx=np.arange(n_ch, n_ch+n_pub)
quick=np.concatenate([dev[:800], pub_idx[:4000]])
for warm in [True, False]:
    t0=time.time()
    m=train(quick, epochs=6, warm=warm)
    p=predict(m, ho)[:,3]
    print(f'  warm={warm}: CYP3A4 holdout rho {spearmanr(y1[ho],p).statistic:.4f} '
          f'({time.time()-t0:.0f}s for 6 epochs)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

## Full run

Public compounds sit in training for every fold; only the challenge rows rotate.

In [ ]:
EPOCHS = 25
RESULTS = []
y_all = {i: meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}
obs_any = np.isfinite(np.column_stack([y_all[i] for i in ISO])).any(1)
dev_rows = np.flatnonzero(obs_any & ~hold); ho_rows = np.flatnonzero(obs_any & hold)
pub_idx = np.arange(n_ch, n_ch + n_pub)

# Checkpoint after every fit, so a disconnect costs one fold rather than the whole run.
# Re-running this cell picks up wherever it stopped.
CKPT = OUT / 'nb5_state.npz'
if CKPT.exists():
    st = np.load(CKPT, allow_pickle=True)
    oof_all = st['oof_all']; done = set(st['done'].tolist())
    hp = st['hp'] if st['hp'].size else None
    tp = st['tp'] if st['tp'].size else None
    print('resuming; folds already done:', sorted(done), '| holdout',
          hp is not None, '| test', tp is not None)
else:
    oof_all = np.full((n_ch, 4), np.nan, np.float32); done = set(); hp = tp = None

def save():
    np.savez_compressed(CKPT, oof_all=oof_all, done=np.array(sorted(done)),
                        hp=hp if hp is not None else np.array([]),
                        tp=tp if tp is not None else np.array([]))

t0 = time.time()
for f, (tr, va) in enumerate(GroupKFold(5).split(dev_rows, dev_rows, G[dev_rows])):
    if f in done:
        continue
    m = train(np.concatenate([dev_rows[tr], pub_idx]), epochs=EPOCHS)
    oof_all[dev_rows[va]] = predict(m, dev_rows[va])[:, :4]
    done.add(f); save()
    print(f'  fold {f} done, checkpointed ({time.time()-t0:.0f}s)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

if hp is None:
    m = train(np.concatenate([dev_rows, pub_idx]), epochs=EPOCHS)
    hp = predict(m, ho_rows)[:, :4]; save()
    print(f'  holdout model done ({time.time()-t0:.0f}s)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

if tp is None:
    mt = train(np.concatenate([dev_rows, ho_rows, pub_idx]), epochs=EPOCHS)
    tp = predict(mt, test=True)[:, :4]; save()
    print(f'  test model done ({time.time()-t0:.0f}s)', flush=True)
    del mt; gc.collect(); torch.cuda.empty_cache()

for j, i in enumerate(ISO):
    y = y_all[i]
    d = dev_rows[np.isfinite(y[dev_rows])]; h = ho_rows[np.isfinite(y[ho_rows])]
    np.savez_compressed(OUT / f'mtl_{i}.npz', oof=oof_all[d, j], dev=d,
                        hold=hp[np.isin(ho_rows, h), j], ho=h, test=tp[:, j])
    r = float(spearmanr(y[d], oof_all[d, j]).statistic)
    rh = float(spearmanr(y[h], hp[np.isin(ho_rows, h), j]).statistic)
    RESULTS.append(dict(iso=i, dev_spearman=r, holdout_spearman=rh))
    json.dump(RESULTS, open(OUT / 'results_nb5.json', 'w'), indent=2)
    print(f'{i}: dev rho {r:.4f} | holdout rho {rh:.4f}', flush=True)
print('\nTabPFN arm for comparison: 0.5881 0.7060 0.4448 0.8115 (dev)')

import shutil
from google.colab import files
shutil.make_archive('/content/notebook5_results', 'zip', OUT)
files.download('/content/notebook5_results.zip')